# Ising Transformer Colab Playground

This notebook clones the repo, lets you configure and run training, and then loads a checkpoint to estimate magnetization and susceptibility from generated samples.

In [ ]:
!git clone https://github.com/labofdoubt/ising-transformer /content/ising-transformer

In [ ]:
%cd /content/ising-transformer
!pip install -q pyyaml tqdm numpy

In [ ]:
from pathlib import Path
import yaml

run_name = "colab_run"
config_path = Path("configs") / f"{run_name}.yaml"

train_setup = {
    "model": {
        "L": 32,
        "patch_r": 2,
        "patch_c": 4,
        "hidden_dim": 128,
        "n_heads": 4,
        "n_blocks": 1,
        "use_layernorm": True,
        "init_std": 0.02,
        "use_pos_emb": True,
        "use_ap": False,
        "beta": 0.44068679350977147,
        "J": 1.0,
        "dtype": "float32",
        "device": "cuda",
    },
    "train": {
        "batch_size": 1024,
        "val_batch_size": 8192,
        "adam_betas": [0.9, 0.95],
        "weight_decay": 0.0,
        "learning_rate": 1e-3,
        "total_steps": 2000,
        "use_cosine_scheduler": False,
        "validate_every_n": 100,
        "save_logs_every_n": 100,
        "save_checkpoint_every_n": 500,
        "resume_checkpoint": None,
        "log_dir": f"runs/{run_name}",
        "checkpoint_dir": f"checkpoints/{run_name}",
        "seed": 1234,
        "grad_clip": None,
    },
}

config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(train_setup, sort_keys=False))
print(config_path)
print(config_path.read_text())

In [ ]:
resume_checkpoint = None

cmd = f"python scripts/train.py --config {config_path}"
if resume_checkpoint:
    cmd += f" --resume {resume_checkpoint}"
print(cmd)
!{cmd}

## Load a checkpoint and estimate magnetization / susceptibility

The definitions used below are:

- `M = (1 / N) * sum_i sigma_i`
- `<|M|>` estimated from generated samples
- `chi = beta * N * (<M^2> - <|M|>^2)`

In [ ]:
import math
import torch

from tvan.ap import ApproximateProbabilityBias
from tvan.checkpoint import load_checkpoint
from tvan.config import load_config
from tvan.generation import generate
from tvan.model import TVANTransformer

checkpoint_path = "checkpoints/colab_run/step_500.pt"
num_samples = 20000
sample_batch_size = 2000

model_cfg, train_cfg = load_config(config_path)
device = torch.device(model_cfg.device)

model = TVANTransformer(model_cfg).to(device)
load_checkpoint(checkpoint_path, model, map_location=device)
model.eval()
ap_module = ApproximateProbabilityBias(model_cfg).to(device) if model_cfg.use_ap else None

@torch.no_grad()
def estimate_magnetization_and_susceptibility(model, model_cfg, checkpoint_path, num_samples, batch_size, ap_module=None):
    magnetizations = []
    num_steps = math.ceil(num_samples / batch_size)

    for step in range(num_steps):
        current_batch = min(batch_size, num_samples - step * batch_size)
        _, _, spins = generate(model, current_batch, model_cfg, ap_module=ap_module, return_spins=True)
        assert spins is not None
        m = spins.float().mean(dim=(1, 2))
        magnetizations.append(m.cpu())

    magnetizations = torch.cat(magnetizations, dim=0)
    avg_abs_m = magnetizations.abs().mean().item()
    avg_m2 = magnetizations.square().mean().item()
    num_spins = model_cfg.L * model_cfg.L
    chi = model_cfg.beta * num_spins * (avg_m2 - avg_abs_m ** 2)
    return {
        "num_samples": int(magnetizations.shape[0]),
        "avg_abs_magnetization": avg_abs_m,
        "avg_m2": avg_m2,
        "susceptibility": chi,
    }

metrics = estimate_magnetization_and_susceptibility(
    model=model,
    model_cfg=model_cfg,
    checkpoint_path=checkpoint_path,
    num_samples=num_samples,
    batch_size=sample_batch_size,
    ap_module=ap_module,
)

for key, value in metrics.items():
    print(f"{key}={value}")